# **Deteksi Pesan Phishing Berhadiah (Smishing) Bahasa Indonesia Berbasis NLP**

**Tema : Inclusive & Resilient Communities**

**Sumber Dataset :**

| No | Dataset | Sumber | Isi | Label |
|----|---|---|---|---|
| 1  | `dataset_sms_spam_v2.csv` | GitHub Gist (yudiwbs) | SMS spam/penipuan Indonesia | spam/promo/penipuan |
| 2  | `indonesia-sms-spam-dataset` | GitHub (bopbi) | SMS spam Indonesia per kategori | prize/loan/evil-service/dll |
| 3  | `indonesian-email-spam` | Kaggle (gevabriel) | Email spam Bahasa Indonesia | spam/ham |

### Link Download Dataset
- **Dataset 1 (GitHub Gist):** https://gist.github.com/agtbaskara/a1a7017027cc1df9d35cf06e1e5575b7
- **Dataset 2 (GitHub bopbi):** https://github.com/bopbi/indonesia-sms-spam-dataset
- **Dataset 3 (Kaggle):** https://www.kaggle.com/datasets/gevabriel/indonesian-email-spam


# **Business Questions**

Sebelum memulai proses analisis data, tim mendefinisikan pertanyaan bisnis yang **terukur** sebagai panduan seluruh proses eksplorasi dan pemodelan.

| No | Pertanyaan Bisnis | Metrik Ukur |
|----|---|---|
|  1 | Seberapa akurat model dalam membedakan pesan phishing dan pesan normal berbahasa Indonesia? | Accuracy, F1-Score, AUC-ROC |
|  2 | Modus phishing apa yang paling sering muncul di Indonesia? | Frekuensi sub-kategori |
|  3 | Apakah keberadaan URL dan nomor panjang menjadi indikator kuat pesan phishing? | Korelasi fitur vs label |
|  4 | Kata dan frasa apa yang paling sering digunakan dalam pesan phishing berhadiah? | Word frequency, WordCloud |
|  5 | Apakah panjang teks pesan phishing berbeda signifikan dengan pesan normal? | Distribusi panjang teks |

> **Catatan:** Pertanyaan-pertanyaan ini akan dijawab secara sistematis melalui proses EDA dan evaluasi model pada notebook selanjutnya.


# **Instalasi Library**

Library yang digunakan dalam pipeline ini:
- **pandas, numpy** — manipulasi dan analisis data tabular
- **PySastrawi** — stemming dan stopword removal khusus Bahasa Indonesia
- **langdetect** — deteksi bahasa otomatis untuk filter kualitas data
- **scikit-learn** — train/val/test split dengan stratifikasi
- **tqdm** — progress bar untuk proses iteratif yang panjang
- **requests** — download dataset dari URL secara otomatis

In [ ]:
!pip install pandas numpy requests PySastrawi langdetect tqdm scikit-learn openpyxl -q
print('✅ Instalasi selesai')

# **Import Library**

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import requests
import warnings
from collections import Counter
from tqdm import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
DetectorFactory.seed = 42    # ← pindah ke sini
tqdm.pandas()
print('✅ Semua library berhasil diimport')

# **Gathering Data**

Proses pengumpulan data dilakukan dari **3 sumber berbeda** untuk memastikan keberagaman data dan representasi modus phishing yang lebih lengkap. Setiap sumber dipilih karena relevansinya dengan konteks SMS/WA phishing Bahasa Indonesia.

| No | Sumber | URL | Isi | Estimasi Data |
|----|---|---|---|---|
| 1  | GitHub Gist (yudiwbs) | [Link](https://gist.github.com/agtbaskara/a1a7017027cc1df9d35cf06e1e5575b7) | SMS spam/penipuan Indonesia | ~574 baris |
| 2  | GitHub (bopbi) | [Link](https://github.com/bopbi/indonesia-sms-spam-dataset) | SMS spam per kategori (prize, loan, dll) | ~288 baris |
| 3  | Kaggle (gevabriel) | [Link](https://www.kaggle.com/datasets/gevabriel/indonesian-email-spam) | Email spam Bahasa Indonesia | ~2.636 baris |

> **Alasan pemilihan sumber data:**
* Sumber data seluruhnya berbahasa Indonesia
* Memiliki label yang dapat dipetakan ke binary phishing/normal
* Isi sumber data mencakup berbagai modus penipuan digital yang relevan dengan konteks Indonesia.


# **Load Dataset 1: SMS Spam Indonesia (GitHub Gist yudiwbs)**

Dataset ini berisi SMS spam Bahasa Indonesia asli yang dikumpulkan dari pengguna Indonesia dengan dua label yaitu `promo` dan `penipuan`

**Alasan Memilih menggunakan dua label :**
- `penipuan` → **1 (phishing)** merupakan pesan berbahaya yang merugikan
- `promo` → **0 (normal)** merupakan pesan promosi operator yang sah

In [ ]:
# Download otomatis via raw URL
URL_GIST = 'https://gist.githubusercontent.com/agtbaskara/a1a7017027cc1df9d35cf06e1e5575b7/raw/dataset_sms_spam_v2.csv'

try:
    df_yudiwbs = pd.read_csv(URL_GIST)
    print(f'✅ Berhasil load dari URL')
except Exception as e:
    print(f'⚠️ Gagal load dari URL: {e}')

    # Fallback: load dari file lokal jika sudah didownload manual
    df_yudiwbs = pd.read_csv('dataset_sms_spam_v2.csv')
    print(f'✅ Berhasil load dari file lokal')

print(f'Shape awal: {df_yudiwbs.shape}')
print(f'Kolom: {df_yudiwbs.columns.tolist()}')
print(f'\nDistribusi label asli:')
print(df_yudiwbs.iloc[:, 1].value_counts())

print(f'\nContoh data:')
df_yudiwbs.head(5)

In [ ]:
# Standardisasi kolom dan label
df_yudiwbs.columns = ['teks', 'label_asli']

# Mapping label:
# penipuan → 1 (phishing/spam berbahaya)
# promo    → 0 (bukan phishing, pesan promosi operator)
# Catatan: untuk penelitian phishing berhadiah, kami fokus pada label 'penipuan'

label_map = {
    'penipuan': 1,
    'promo':    0
}
df_yudiwbs['label'] = df_yudiwbs['label_asli'].str.strip().str.lower().map(label_map)
df_yudiwbs['sumber'] = 'yudiwbs_gist'

# Hapus yang labelnya tidak dikenali
df_yudiwbs = df_yudiwbs.dropna(subset=['label']).reset_index(drop=True)

print(f'Dataset 1 setelah standarisasi (mapping):')
print(df_yudiwbs['label'].value_counts())
print(f'        → 1 = phishing/penipuan, 0 = normal/promo')
df_yudiwbs[['teks', 'label', 'sumber']].head(3)

# **Load Dataset 2: Indonesia SMS Spam Dataset (GitHub bopbi)**

Dataset ini berisi SMS spam Indonesia yang dikategorikan per jenis penipuan dalam bentuk file teks terpisah di folder, yaitu  
`prize` (berhadiah), `loan` (pinjaman), `evil-service`, `non-provider-promo`, `premium-sms`

**Labeling per kategori :**
- `prize`, `loan`, `evil-service`, `premium-sms` → **1 (phishing)** merupakan modus penipuan aktif
- `non-provider-promo` → **0 (normal)** merupakan promosi yang tidak berbahaya

> **Cara mendapatkan dataset:** Download ZIP dari [GitHub bopbi](https://github.com/bopbi/indonesia-sms-spam-dataset) lalu upload ke Colab menggunakan cell berikut.

In [ ]:
from google.colab import files

# Ambil nama file
zip_name = 'indonesia-sms-spam-dataset-main.zip'

# Extract
print(f'\nExtracting: {zip_name}...')
!unzip -q "$zip_name"
print('✅ Ekstraksi selesai')

# Cek folder
print("\nIsi folder:")
print(os.listdir())

In [ ]:
FOLDER_BOPBI = 'bopbi dataset indonesia-sms-spam-dataset-main'
KATEGORI_PHISHING = {'prize', 'loan', 'evil-service', 'premium-sms'}

df_bopbi = pd.DataFrame()
records   = []

if os.path.exists(FOLDER_BOPBI):
    for subfolder in os.listdir(FOLDER_BOPBI):
        path_sub = os.path.join(FOLDER_BOPBI, subfolder)
        if not os.path.isdir(path_sub):
            continue

        label_val = 1 if subfolder.lower() in KATEGORI_PHISHING else 0

        for fname in os.listdir(path_sub):
            fpath = os.path.join(path_sub, fname)
            try:
                # ✅ FIX 1: Baca seluruh file sebagai 1 SMS utuh
                # Bug lama: `for line in f` — memecah 1 SMS menjadi banyak fragmen baris
                # Fix: f.read() → 1 file = 1 SMS = 1 baris di dataframe
                with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                    pesan = f.read().strip()
                if pesan:
                    records.append({
                        'teks'  : pesan,
                        'label' : label_val,
                        'sumber': f'bopbi_{subfolder}'
                    })
            except Exception as e:
                print(f'Gagal baca {fpath}: {e}')

    if records:
        df_bopbi = pd.DataFrame(records)
        print(f'✅ Load berhasil: {len(df_bopbi)} pesan')
        print(f'   (Sebelumnya: ~288 fragmen baris → sekarang: {len(df_bopbi)} SMS utuh)')
        print('\nDistribusi per subfolder:')
        print(df_bopbi['sumber'].value_counts())
        print('\nDistribusi label:')
        print(df_bopbi['label'].value_counts())
        print('    → 1 = phishing, 0 = normal')
        print('\nSample panjang teks (validasi):')
        print(df_bopbi['teks'].str.len().describe().round(0))
    else:
        print('Tidak ada data yang terbaca.')
else:
    print(f'Folder {FOLDER_BOPBI} tidak ditemukan.')

# **Load Dataset 3: Indonesian Email Spam (Kaggle)**

Dataset ini berisi email spam Bahasa Indonesia dari Kaggle dengan label `spam` dan `ham`.

**Labeling:**
- `spam` → **1 (phishing)** merupakan pesan tidak diinginkan yang berpotensi berbahaya
- `ham` → **0 (normal)** merupakan pesan sah

> **Cara mendapatkan dataset:** Jalankan cell berikut menggunakan Kaggle API, atau download manual dari [link ini](https://www.kaggle.com/datasets/gevabriel/indonesian-email-spam).

In [ ]:
# Kaggle API (jika sudah setup)
!kaggle datasets download -d gevabriel/indonesian-email-spam
!unzip indonesian-email-spam.zip
print('✅ Download via Kaggle API berhasil')

In [ ]:
df_email     = pd.DataFrame()
df_email_std = pd.DataFrame(columns=['teks', 'label', 'sumber'])

kandidat_file = [
    'email_spam_indo.csv',
    'indonesian_email_spam.csv',
    'email_spam_indonesia.csv',
    'spam_email.csv',
    'dataset.csv'
]

for fname in kandidat_file:
    if os.path.exists(fname):
        df_email = pd.read_csv(fname)
        print(f'✅ Berhasil load: {fname}')
        print(f'Shape  : {df_email.shape}')
        print(f'Kolom  : {df_email.columns.tolist()}')
        print(df_email.head(3))
        break

if df_email.empty:
    print('File tidak ditemukan — upload manual file CSV ke Colab')

In [ ]:
if not df_email.empty:
    KOLOM_TEKS  = 'Pesan'
    KOLOM_LABEL = 'Kategori'

    df_email_std            = df_email[[KOLOM_TEKS, KOLOM_LABEL]].copy()
    df_email_std.columns    = ['teks', 'label_asli']

    email_label_map = {
        'spam': 1, 'ham': 0,
        'normal': 0, '1': 1, '0': 0,
        1: 1, 0: 0
    }
    df_email_std['label']  = df_email_std['label_asli'].astype(str).str.lower().map(email_label_map)
    df_email_std['sumber'] = 'kaggle_email_indo'
    df_email_std           = df_email_std.dropna(subset=['label']).reset_index(drop=True)

    print(f'✅ Dataset email setelah standardisasi: {df_email_std.shape}')
    print('\nDistribusi label:')
    print(df_email_std['label'].value_counts())
    df_email_std.head(3)

# **✅ FIX 2: Filter Email Korporat Out-of-Domain**

Dataset Kaggle mencakup email korporat panjang (rata-rata 1.200+ karakter) yang secara domain sangat berbeda dari SMS/WA phishing Indonesia. Email ini menyebabkan model 'bingung' karena:
- Pola bahasa formal email ≠ pola bahasa SMS phishing
- Panjang rata-rata 1.200+ kar vs SMS normal ~50–300 kar
- Probabilitas model menjadi tanggung (0.2–0.4) untuk sampel ini

**Solusi:** Hapus sampel dengan panjang teks > 800 karakter dari dataset Kaggle.


In [ ]:
# ✅ FIX 2: Filter email korporat out-of-domain
# Email panjang (>800 karakter) secara domain sangat berbeda dari SMS/WA phishing Indonesia.
# Pola bahasa, panjang teks, dan struktur kalimat tidak selaras → menyebabkan model bingung.
# Threshold 800 karakter dipilih berdasarkan distribusi panjang SMS/WA Indonesia (~50–300 kar).

if not df_email_std.empty:
    sebelum = len(df_email_std)
    panjang_email = df_email_std['teks'].str.len()

    print('Distribusi panjang teks email (sebelum filter):')
    print(panjang_email.describe().round(0))

    # Filter: hapus email > 800 karakter (email korporat panjang)
    df_email_std = df_email_std[panjang_email <= 800].reset_index(drop=True)

    sesudah = len(df_email_std)
    print(f'''
✅ Filter email korporat selesai:
   Sebelum filter : {sebelum} baris
   Dibuang        : {sebelum - sesudah} baris (email korporat >800 kar)
   Sisa           : {sesudah} baris (konteks SMS-like)''')
    print('\nDistribusi label setelah filter:')
    print(df_email_std['label'].value_counts())


# **Merge Semua Dataset**

In [ ]:
# Siapkan kolom seragam dari semua sumber
kolom_final = ['teks', 'label', 'sumber']
frames = []

# Dataset 1: yudiwbs
if 'df_yudiwbs' in dir() and not df_yudiwbs.empty:
    frames.append(df_yudiwbs[kolom_final])
    print(f'Dataset yudiwbs  : {len(df_yudiwbs):>5} baris')

# Dataset 2: bopbi
if 'df_bopbi' in dir() and not df_bopbi.empty:
    frames.append(df_bopbi[['teks', 'label', 'sumber']])
    print(f'Dataset bopbi    : {len(df_bopbi):>5} baris')

# Dataset 3: email
if 'df_email_std' in dir() and not df_email_std.empty:
    frames.append(df_email_std[kolom_final])
    print(f'Dataset email    : {len(df_email_std):>5} baris')

# Menggabungkan dataset
if frames:
    df_gabung          = pd.concat(frames, ignore_index=True)
    df_gabung['label'] = df_gabung['label'].astype(int)

    print(f'\n{"="*45}')
    print(f'HASIL MERGE')
    print(f'{"="*45}')
    print(f'Total baris        : {len(df_gabung)}')
    print(f'\nDistribusi label:')
    print(df_gabung['label'].value_counts())
    print(f'  → 1 = phishing, 0 = normal')
    print(f'\nDistribusi sumber:')
    print(df_gabung['sumber'].value_counts())
else:
    print('Tidak ada dataset yang berhasil digabung')

    print('Tidak ada dataset yang berhasil digabung')

# **Assessing Data**

Pada tahap ini dilakukan evaluasi kualitas dan struktur data hasil merge untuk mengidentifikasi masalah yang perlu ditangani sebelum cleaning.

**Adapun hal yang akan diperiksa:**
1. Missing values per kolom
2. Duplikat data
3. Distribusi label (imbalance check)
4. Panjang teks minimum dan maksimum
5. Konsistensi format label


In [ ]:
print('='*50)
print('ASSESSING DATA — HASIL MERGE')
print('='*50)

print(f'\n[1] Shape dataset     : {df_gabung.shape}')

print(f'\n[2] Missing values per kolom:')
mv = df_gabung.isnull().sum()
for col, val in mv.items():
    status = '✅ tidak ada' if val == 0 else f'⚠️  {val} baris ({val/len(df_gabung)*100:.1f}%)'
    print(f'    {col:<10}: {status}')

print(f'\n[3] Duplikat:')
dup = df_gabung.duplicated(subset='teks').sum()
print(f'    Jumlah duplikat teks : {dup} baris ({dup/len(df_gabung)*100:.1f}%)')

print(f'\n[4] Distribusi label:')
vc = df_gabung['label'].value_counts()
total = len(df_gabung)
for lbl, cnt in vc.items():
    nama = 'phishing' if lbl == 1 else 'normal'
    print(f'    Label {lbl} ({nama:<8}) : {cnt:>5} baris ({cnt/total*100:.1f}%)')
rasio = vc.get(0,0) / vc.get(1,1)
print(f'    Rasio normal:phishing = {rasio:.2f}:1')
if rasio < 0.5 or rasio > 2:
    print(f'    ⚠️  Data imbalanced — perlu penanganan saat modelling')
else:
    print(f'    ✅ Distribusi label cukup seimbang')

print(f'\n[5] Statistik panjang teks:')
panjang = df_gabung['teks'].str.len()
print(f'    Rata-rata  : {panjang.mean():.0f} karakter')
print(f'    Minimum    : {panjang.min()} karakter')
print(f'    Maksimum   : {panjang.max()} karakter')
if panjang.min() < 10:
    print(f'    ⚠️  Ada teks sangat pendek (<10 karakter) — perlu difilter')

print(f'\n[6] Tipe data kolom:')
print(df_gabung.dtypes)


Berdasarkan hasil pemeriksaan di atas, ditemukan beberapa hal yang perlu ditangani pada tahap cleaning:

1. **Missing values** — tidak ditemukan missing values pada seluruh kolom (`teks`, `label`, `sumber`), sehingga tidak diperlukan penanganan khusus untuk tahap ini.

2. **Duplikat teks** — ditemukan **41 baris duplikat (1.2%)** dari total 3.498 baris. Duplikat akan dihapus untuk menghindari bias model akibat pengulangan data yang sama.

3. **Teks terlalu pendek** — ditemukan teks dengan panjang minimum **3 karakter**, jauh di bawah ambang batas 10 karakter yang dianggap informatif. Teks seperti ini tidak mengandung cukup informasi untuk dianalisis dan akan difilter.

4. **Deteksi bahasa** — dari 3 sumber yang digabungkan, tidak semua teks dijamin berbahasa Indonesia. Beberapa data dari dataset email kemungkinan mengandung teks berbahasa Inggris atau bahasa lain yang di luar scope penelitian. Filter bahasa menggunakan `langdetect` akan diterapkan, dengan pelonggaran untuk dataset email yang memperbolehkan bahasa Inggris karena konteks bisnis.

5. **Distribusi label** — distribusi label cukup seimbang dengan rasio **normal:phishing = 0.76:1** (56.7% phishing vs 43.3% normal). Kondisi ini baik untuk training model binary classification karena model tidak akan bias ke salah satu kelas secara signifikan.

# **Cleaning Data**

Tahap cleaning dilakukan secara bertahap dengan justifikasi yang jelas di setiap langkah. Proses cleaning mencakup:

1. **Normalisasi teks** — standardisasi URL, nomor telepon, kode OTP menjadi token khusus agar model dapat menangkap pola ini secara generik
2. **Filter kualitas bahasa** — memastikan semua data dalam Bahasa Indonesia atau Melayu
3. **Hapus duplikat** — menghindari data bias akibat pengulangan pesan yang sama
4. **Filter panjang minimum** — membuang teks yang terlalu pendek dan tidak informatif


# **Cleaning & Filter Kualitas**

In [ ]:
import re
from langdetect import detect, LangDetectException
from tqdm.auto import tqdm
tqdm.pandas()

def bersihkan_teks(teks):
    teks = str(teks).strip()
    teks = re.sub(r'http\S+|www\.\S+', '[URL]', teks)
    teks = re.sub(r'\b\d{10,}\b', '[NOMOR_PANJANG]', teks)
    teks = re.sub(
        r'(?i)\b(kode|verifikasi|otp|pin|token|nomor|no|code|verification)([\s:]*)\d{4,6}\b',
        r'\1\2[KODE]',
        teks
    )
    teks = re.sub(r'<[^>]+>', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

def filter_kualitas(teks, sumber=''):
    teks = str(teks).strip()
    if len(teks) < 10:
        return False
    try:
        lang = detect(teks)
        if sumber == 'kaggle_email_indo':
            return lang in ('id', 'ms', 'en')
        return lang in ('id', 'ms')
    except LangDetectException:
        return False

print('Membersihkan teks...')
df_gabung['teks_bersih'] = df_gabung['teks'].apply(bersihkan_teks)
print('✅ Cleaning selesai')

print('\nMemfilter kualitas bahasa (~1-2 menit)...')
df_gabung['lolos_filter'] = df_gabung.progress_apply(
    lambda row: filter_kualitas(row['teks_bersih'], row['sumber']), axis=1
)
print('✅ Filter selesai')

# Verifikasi
print('\nKolom:', df_gabung.columns.tolist())
print(df_gabung['lolos_filter'].value_counts())

In [ ]:
# Jalankan dulu cell diagnostik ini
print(df_gabung.columns.tolist())
print(df_gabung.shape)

In [ ]:
# Terapkan filter dan hapus duplikat
sebelum_filter = len(df_gabung)
df_bersih      = df_gabung[df_gabung['lolos_filter']].copy()

sebelum_dedup  = len(df_bersih)
df_bersih      = df_bersih.drop_duplicates(subset='teks_bersih').reset_index(drop=True)

print('='*50)
print('RINGKASAN PROSES CLEANING')
print('='*50)
print(f'Data sebelum filter   : {sebelum_filter:>6} baris')
print(f'Data setelah filter   : {sebelum_dedup:>6} baris  (-{sebelum_filter-sebelum_dedup} dibuang, bukan Bahasa Indonesia)')
print(f'Data setelah dedup    : {len(df_bersih):>6} baris  (-{sebelum_dedup-len(df_bersih)} duplikat dihapus)')
print(f'\nDistribusi label setelah cleaning:')
vc = df_bersih['label'].value_counts()
for lbl, cnt in vc.items():
    nama = 'phishing' if lbl == 1 else 'normal'
    print(f'  Label {lbl} ({nama:<8}) : {cnt} baris ({cnt/len(df_bersih)*100:.1f}%)')


# **Preprocessing Teks Bahasa Indonesia**
Preprocessing dilakukan untuk mengubah teks mentah menjadi representasi yang lebih bersih dan konsisten sebelum digunakan dalam pemodelan. Tahap ini menggunakan library **PySastrawi** yang dirancang khusus untuk Bahasa Indonesia.

**Pipeline preprocessing:**
1. **Lowercase** — standardisasi huruf kecil semua
2. **Pisahkan token penting** — [URL], [NOMOR_PANJANG], [KODE] dijaga agar tidak hilang
3. **Hapus karakter non-alfabet** — angka dan simbol dihapus (kecuali token khusus)
4. **Stopword removal** — hapus kata umum yang tidak bermakna (PySastrawi + custom)
5. **Stemming** — ubah kata ke bentuk dasar (berlari → lari, menang → menang)
6. **Gabungkan kembali token penting** — [URL], [NOMOR_PANJANG], [KODE] ditambahkan kembali

Di mana, Bahasa Indonesia memiliki banyak imbuhan (me-, di-, ber-, -kan, -an) yang membuat variasi kata sangat banyak. Sehingga, Stemming membantu model mengenali kata yang sama dalam bentuk berbeda.


In [ ]:
# Setup PySastrawi
factory_stem = StemmerFactory()
factory_stop = StopWordRemoverFactory()
stemmer      = factory_stem.create_stemmer()
stop_remover = factory_stop.create_stop_word_remover()

# Stopword tambahan khas kpnteks SMS/WA
STOPWORD_TAMBAHAN = {
    'yth', 'bpk', 'ibu', 'dear', 'customer', 'pelanggan',
    'dpt', 'utk', 'dgn', 'dlm', 'sdh', 'tdk', 'hrs',
    'anda', 'kamu', 'saya', 'nya', 'yang', 'dan', 'atau',
    'ini', 'itu', 'dengan', 'untuk', 'dari', 'ke', 'di'
}

# Regex token penting — sinkron dengan bersihkan_teks
_RE_TOKEN_PENTING = re.compile(r'\[(url|nomor_panjang|kode)\]', re.IGNORECASE)

def preprocess(teks):
    """Pipeline preprocessing teks SMS/WA Bahasa Indonesia."""

    # 1. Lowercase : standardisasi agar 'Selamat' == 'selamat
    teks = teks.lower()

    # 2. Pisahkan token penting sebelum cleaning, di mana [URL] dll harus tetap ada karena informatif
    tokens_penting = _RE_TOKEN_PENTING.findall(teks)
    tokens_penting_str = ['[' + t + ']' for t in tokens_penting]
    teks_tanpa_token = _RE_TOKEN_PENTING.sub('', teks)

    # 3. Hapus karakter non-alfabet (kecuali spasi) karena angka bebas dan simbol tidak membantu model
    teks_clean = re.sub(r'[^a-z\s]', ' ', teks_tanpa_token)
    teks_clean = re.sub(r'\s+', ' ', teks_clean).strip()

    # 4. Hapus stopword PySastrawi
    teks_clean = stop_remover.remove(teks_clean)

    # 5. Hapus stopword tambahan
    words = [w for w in teks_clean.split() if w not in STOPWORD_TAMBAHAN]

    # 6. Stemming : mengurangi variasi morfologi Bahasa Indonesia
    words = [stemmer.stem(w) for w in words if w]

    # 7. Gabungkan kembali dengan token penting
    hasil = ' '.join(words + tokens_penting_str)
    return hasil.strip()

print('Menjalankan preprocessing...')
df_bersih['teks_processed'] = df_bersih['teks_bersih'].progress_apply(preprocess)
print('✅ Preprocessing selesai')

# Contoh hasil preprocessing
print('\n=== Contoh Hasil Preprocessing ===')
for label_val, nama in [(1, 'PHISHING'), (0, 'NORMAL')]:
    subset = df_bersih[df_bersih['label'] == label_val]
    if subset.empty:
        print(f'\n[{nama}] — tidak ada data')
        continue
    sample = subset.iloc[0]
    print(f'\n[{nama}]')
    print(f'  Asli      : {sample["teks"][:150]}')
    print(f'  Bersih    : {sample["teks_bersih"][:150]}')
    print(f'  Processed : {sample["teks_processed"][:150]}')

# **Feature Engineering**

Selain fitur teks, kami menambahkan fitur numerik yang dapat membantu model mendeteksi pola phishing. Fitur-fitur ini dibuat berdasarkan **pengetahuan domain** tentang ciri khas pesan phishing Indonesia.

| Fitur Baru | Tipe | Justifikasi |
|---|---|---|
| `text_length` | int | Pesan phishing cenderung lebih panjang (banyak bujukan) |
| `word_count` | int | Jumlah kata sebagai proxy kompleksitas pesan |
| `exclamation_count` | int | Tanda seru berlebihan adalah ciri khas clickbait/phishing |
| `has_url` | int | Keberadaan URL adalah indikator kuat phishing |
| `has_phone` | int | Nomor telepon sering ada di pesan penipuan |
| `has_code` | int | Kode OTP palsu sering dipakai di smishing |
| `uppercase_ratio` | float | Huruf kapital berlebihan = penekanan/urgensi palsu |
| `sub_kategori` | str | Jenis modus phishing berdasarkan keyword matching |


In [ ]:
# Kata kunci khas modus phishing berhadiah Indonesia
POLA_PHISHING = {
    'berhadiah'  : ['selamat', 'menang', 'terpilih', 'hadiah', 'undian',
                    'pemenang', 'mobil', 'motor', 'uang tunai', 'juta'],
    'pinjaman'   : ['pinjaman', 'kredit', 'kta', 'bunga', 'jaminan',
                    'dana tunai', 'bpkb', 'syarat mudah'],
    'urgensi'    : ['segera', 'buruan', 'jatuh tempo', 'sebelum', 'hari ini',
                    'terbatas', 'cepat', 'langsung'],
    'sosial_eng' : ['mama', 'papa', 'ayah', 'bapak', 'nomor baru',
                    'masalah', 'tolong', 'kirimin pulsa'],
    'phishing_url': ['[url]', 'klik', 'blogspot', 'verifikasi', 'website'],
    'minta_data' : ['pin', 'rekening', 'transfer', 'rek', 'a/n', 'bca',
                    'bri', 'bni', 'mandiri', 'pulsa',
                    '[nomor_panjang]', '[kode]'],
}

def deteksi_kategori(teks):
    teks_lower = str(teks).lower()
    hasil = [kat for kat, kw_list in POLA_PHISHING.items()
             if any(kw in teks_lower for kw in kw_list)]
    return '|'.join(hasil) if hasil else 'lainnya'

# Buat fitur numerik
df_bersih['text_length']       = df_bersih['teks'].str.len()
df_bersih['word_count']        = df_bersih['teks'].str.split().str.len()
df_bersih['exclamation_count'] = df_bersih['teks'].str.count('!')
df_bersih['has_url']           = df_bersih['teks_bersih'].str.contains(r'\[URL\]',    case=False).astype(int)
df_bersih['has_phone']         = df_bersih['teks_bersih'].str.contains(r'\[NOMOR_PANJANG\]', case=False).astype(int)
df_bersih['has_code']          = df_bersih['teks_bersih'].str.contains(r'\[KODE\]',   case=False).astype(int)
df_bersih['uppercase_ratio']   = df_bersih['teks'].apply(
    lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)
)

# Sub kategori (hanya untuk phishing)
df_bersih['sub_kategori'] = 'normal'
mask_phishing = df_bersih['label'] == 1
df_bersih.loc[mask_phishing, 'sub_kategori'] = (
    df_bersih.loc[mask_phishing, 'teks_bersih'].apply(deteksi_kategori)
)

print('✅ Feature engineering selesai')
print(f'\nFitur baru yang ditambahkan:')
fitur_baru = ['text_length','word_count','exclamation_count',
              'has_url','has_phone','has_code','uppercase_ratio','sub_kategori']
for f in fitur_baru:
    print(f'  + {f}')

print(f'\nDistribusi Sub-Kategori Phishing:')
semua_kat = []
for k in df_bersih[mask_phishing]['sub_kategori']:
    semua_kat.extend(k.split('|'))
for kat, cnt in Counter(semua_kat).most_common():
    print(f'  {kat:<20}: {cnt} pesan')


# **Data Dictionary**

Berikut adalah penjelasan lengkap setiap kolom pada dataset final yang akan digunakan untuk EDA dan pemodelan.

| Kolom | Tipe | Deskripsi | Contoh Nilai |
|---|---|---|---|
| `teks` | str | Teks SMS/WA/email asli tanpa perubahan apapun | "Selamat! Anda terpilih mendapat hadiah..." |
| `teks_bersih` | str | Teks setelah cleaning dasar (URL/nomor diganti token) | "Selamat! Anda terpilih [URL]" |
| `teks_processed` | str | Teks setelah preprocessing penuh (stemming, stopword removal) | "selamat pilih hadiah [url]" |
| `label` | int | Target klasifikasi binary | 1 (phishing), 0 (normal) |
| `sumber` | str | Asal dataset | "yudiwbs_gist", "bopbi_prize", "kaggle_email_indo" |
| `text_length` | int | Panjang teks asli dalam karakter | 245 |
| `word_count` | int | Jumlah kata dalam teks asli | 38 |
| `exclamation_count` | int | Jumlah tanda seru dalam teks asli | 3 |
| `has_url` | int | 1 jika teks mengandung URL, 0 jika tidak | 1 |
| `has_phone` | int | 1 jika teks mengandung nomor telepon/rekening panjang | 0 |
| `has_code` | int | 1 jika teks mengandung kode OTP (4-6 digit) | 0 |
| `uppercase_ratio` | float | Proporsi huruf kapital dari total karakter | 0.12 |
| `sub_kategori` | str | Jenis modus phishing yang terdeteksi (hanya untuk label=1) | "berhadiah\|urgensi" |

**Keterangan label:**
- `1` = Pesan phishing / smishing / penipuan yang berpotensi merugikan
- `0` = Pesan normal / sah (promo operator, email ham, dll)


# **Validasi & Split Dataset**
Sebelum export data, dilakukan validasi menyeluruh untuk memastikan dataset sudah benar-benar siap digunakan untuk EDA dan pemodelan.

In [ ]:
# Filter teks_processed terlalu pendek
sebelum = len(df_bersih)
df_bersih = df_bersih[df_bersih['teks_processed'].str.len() > 5].reset_index(drop=True)
print(f'Baris dibuang (processed < 5 karakter): {sebelum - len(df_bersih)}')

# Kolom output final
kolom_output = [
    'teks', 'teks_bersih', 'teks_processed',
    'label', 'sub_kategori', 'sumber',
    'text_length', 'word_count', 'exclamation_count',
    'has_url', 'has_phone', 'has_code', 'uppercase_ratio'
]
df_final = df_bersih[kolom_output].copy()

print('='*55)
print('VALIDASI DATASET FINAL')
print('='*55)
print(f'Total baris            : {len(df_final)}')

print(f'\nMissing values:')
mv = df_final.isnull().sum()
if mv.sum() == 0:
    print('  ✅ Tidak ada missing values')
else:
    print(mv[mv > 0])

print(f'\nDistribusi label:')
vc = df_final['label'].value_counts()
for lbl, cnt in vc.items():
    nama = 'phishing' if lbl == 1 else 'normal'
    print(f'  Label {lbl} ({nama:<8}) : {cnt} ({cnt/len(df_final)*100:.1f}%)')
rasio = vc.get(0,0) / vc.get(1,1)
print(f'  Rasio normal:phishing = {rasio:.2f}:1')

print(f'\nStatistik panjang teks asli:')
print(f'  Rata-rata  : {df_final["text_length"].mean():.0f} karakter')
print(f'  Minimum    : {df_final["text_length"].min()} karakter')
print(f'  Maksimum   : {df_final["text_length"].max()} karakter')

print(f'\nStatistik fitur numerik:')
fitur_num = ['exclamation_count','has_url','has_phone','has_code']
print(df_final[fitur_num].describe().round(3))

# Cek data leakage — pastikan tidak ada kolom target di fitur
print(f'\nCek data leakage:')
print(f'  Kolom "label" tidak ada di teks_processed: ', end='')
leakage = df_final['teks_processed'].str.contains('label').any()
print('PERLU DICEK' if leakage else '✅ Aman')

print(f'\n✅ Dataset final siap untuk tahap EDA dan Modelling')

# **Split Dataset & Export**

Dataset dibagi menjadi 3 bagian dengan stratified split.
Split dilakukan **per domain** (SMS dan email dipisah) agar proporsi domain terkontrol di setiap subset.

| Split | Proporsi | Kegunaan |
|---|---|---|
| Train | 80% | Training model |
| Validation | 10% | Tuning hyperparameter |
| Test | 10% | Evaluasi final (tidak boleh dilihat selama training) |

> **✅ FIX 3 & 4 — Cap email per split:**  
> Sebelumnya proporsi email di train=73%, val=67%, test=65% sehingga model belajar dan dievaluasi mayoritas pada domain email korporat, bukan SMS/WA Indonesia.  
> Setelah fix, email dibatasi **max 50%** di semua split (train, val, test) — proporsi seimbang dengan SMS asli.  
> Ubah `EMAIL_CAP_RATIO` ke `0.5` atau `0.4` di cell train jika MAE masih tinggi.


In [ ]:
# Definisi domain SMS
sms_sources = [
    'yudiwbs_gist',
    'bopbi_loan',
    'bopbi_prize',
    'bopbi_premium-sms',
    'bopbi_evil-service',
    'bopbi_non-provider-promo'
]

# Tambah kolom domain
df_final['domain'] = df_final['sumber'].apply(
    lambda x: 'sms' if x in sms_sources else 'email'
)

print(df_final['domain'].value_counts())

In [ ]:
sms_df   = df_final[df_final['domain'] == 'sms'].copy()
email_df = df_final[df_final['domain'] == 'email'].copy()

print(f"SMS   : {len(sms_df)}")
print(f"Email : {len(email_df)}")

In [ ]:
from sklearn.model_selection import train_test_split

# =========================
# SPLIT SMS
# =========================

sms_train, sms_temp = train_test_split(
    sms_df,
    test_size=0.2,
    random_state=42,
    stratify=sms_df['label']
)

sms_val, sms_test = train_test_split(
    sms_temp,
    test_size=0.5,
    random_state=42,
    stratify=sms_temp['label']
)

In [ ]:
# =========================
# SPLIT EMAIL
# =========================

email_train, email_temp = train_test_split(
    email_df,
    test_size=0.2,
    random_state=42,
    stratify=email_df['label']
)

email_val, email_test = train_test_split(
    email_temp,
    test_size=0.5,
    random_state=42,
    stratify=email_temp['label']
)

In [ ]:
# =========================
# TRAIN FINAL
# ✅ FIX 3: Cap proporsi email di train agar tidak dominasi SMS
# =========================

sms_train_df   = sms_train.copy()
email_train_df = email_train.copy()

n_sms   = len(sms_train_df)
n_email = len(email_train_df)

# Batasi email max 50% dari total train (= sama banyak dengan SMS)
# Ubah EMAIL_CAP_RATIO ke 0.4 atau 0.3 jika MAE masih tinggi
EMAIL_CAP_RATIO = 0.4
max_email = int((EMAIL_CAP_RATIO / (1 - EMAIL_CAP_RATIO)) * n_sms)
max_email = min(max_email, n_email)

if n_email > max_email:
    email_train_df = (
        email_train_df
        .groupby('label', group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), int(max_email * len(x) / n_email)),
            random_state=42
        ))
    )

df_train = pd.concat([
    sms_train_df,
    email_train_df
]).sample(frac=1, random_state=42).reset_index(drop=True)

n_email_final = df_train['sumber'].str.contains('kaggle').sum()
n_sms_final   = len(df_train) - n_email_final
print(f'Train total   : {len(df_train)}')
print(f'  SMS asli    : {n_sms_final} ({n_sms_final/len(df_train)*100:.1f}%)')
print(f'  Email       : {n_email_final} ({n_email_final/len(df_train)*100:.1f}%)')
print(f'\nLabel dist:')
print(df_train['label'].value_counts())


In [ ]:
# Validation khusus
df_val_sms   = sms_val.reset_index(drop=True)
df_val_email = email_val.reset_index(drop=True)

# Test khusus
df_test_sms   = sms_test.reset_index(drop=True)
df_test_email = email_test.reset_index(drop=True)

# **Augmentasi Data Training**

Augmentasi dilakukan **hanya pada data training** setelah split untuk menghindari data leakage ke validation dan test set.


**Mengapa augmentasi diperlukan?**
Dataset asli memiliki 2.580 data training (cukup untuk lulus checklist minimum), namun berisiko tidak mencapai target akurasi 85% pada model BiLSTM dari scratch. Augmentasi menambah variasi data training tanpa mencari dataset baru, sehingga model lebih robust terhadap variasi input nyata dari pengguna.

**Teknik yang digunakan:**
| Teknik | Deskripsi | Contoh |
|---|---|---|
| **Sinonim lokal** | Ganti kata dengan padanan khas SMS Indonesia | `anda` → `kamu`, `lo` |
| **Variasi singkatan** | Tambah/hapus singkatan SMS | `untuk` → `utk`, `buat` |
| **Variasi urgensi** | Ganti kata urgensi dengan sinonim | `segera` → `buruan`, `cepat` |
| **Random word drop** | Hapus kata non-penting secara acak | simulasi typo/pesan singkat |

> **Catatan penting:** Augmentasi HANYA diterapkan pada `df_train`. `df_val` dan `df_test` tidak disentuh agar evaluasi model tetap mencerminkan kondisi data nyata.

> **Menghindari Data leakage:** Split dilakukan terlebih dahulu di section sebelumnya, baru augmentasi diterapkan pada `df_train` saja.


In [ ]:
import random
random.seed(42)

# Kamus variasi khas SMS/WA Indonesia
SINONIM_INDONESIA = {
    # Sapaan & subjek
    'anda'      : ['kamu', 'lo', 'saudara', 'bapak', 'ibu'],
    'kamu'      : ['anda', 'lo', 'saudara'],

    # Kata kerja phishing
    'menang'    : ['terpilih', 'beruntung', 'mendapat', 'dapet'],
    'terpilih'  : ['menang', 'beruntung', 'dipilih'],
    'klaim'     : ['ambil', 'dapatkan', 'cairkan', 'tukarkan'],
    'transfer'  : ['kirim', 'setorkan', 'bayar'],
    'hubungi'   : ['kontak', 'wa', 'telp', 'kabari'],

    # Kata benda phishing
    'hadiah'    : ['reward', 'prize', 'bonus', 'kejutan'],
    'pulsa'     : ['kuota', 'saldo', 'kredit'],
    'rekening'  : ['rek', 'akun', 'nomor rekening'],

    # Urgensi
    'segera'    : ['buruan', 'cepat', 'langsung', 'sekarang juga'],
    'buruan'    : ['segera', 'cepat', 'langsung'],
    'cepat'     : ['segera', 'buruan', 'lekas'],
    'sekarang'  : ['saat ini', 'hari ini', 'langsung'],

    # Singkatan vs penuh
    'untuk'     : ['utk', 'buat', 'guna'],
    'utk'       : ['untuk', 'buat'],
    'dengan'    : ['dgn', 'sama', 'bareng'],
    'dgn'       : ['dengan', 'sama'],
    'tidak'     : ['ga', 'gak', 'ngga', 'bukan'],
    'sudah'     : ['udah', 'telah', 'sdh'],
    'bisa'      : ['bs', 'dapat', 'sanggup'],
    'dapat'     : ['dpt', 'bisa', 'mendapat'],
    'dpt'       : ['dapat', 'bisa'],
}

def augmentasi_sinonim(teks, prob=0.4):
    """
    Ganti kata secara acak dengan sinonimnya.
    prob: probabilitas penggantian tiap kata (default 40%)
    """

    words = teks.split()
    for i, word in enumerate(words):
        word_lower = word.lower()
        if word_lower in SINONIM_INDONESIA and random.random() < prob:
            pengganti = random.choice(SINONIM_INDONESIA[word_lower])
            # Pertahankan kapitalisasi awal
            if word[0].isupper():
                pengganti = pengganti.capitalize()
            words[i] = pengganti
    return ' '.join(words)

def augmentasi_singkatan(teks, prob=0.35):
    """
    Konversi kata formal ke singkatan khas SMS Indonesia atau sebaliknya.
    Mensimulasikan variasi penulisan pengguna nyata.
    """

    singkatan_map = {
        'untuk'   : 'utk',   'dengan'  : 'dgn',
        'sudah'   : 'udah',  'tidak'   : 'ga',
        'bisa'    : 'bs',    'dapat'   : 'dpt',
        'yang'    : 'yg',    'karena'  : 'krn',
        'sampai'  : 'sampe', 'begitu'  : 'gitu',
        'seperti' : 'kyk',   'makanya' : 'makanya',
    }
    words = teks.split()
    for i, word in enumerate(words):
        word_lower = word.lower()
        if word_lower in singkatan_map and random.random() < prob:
            words[i] = singkatan_map[word_lower]
    return ' '.join(words)

def augmentasi_random_drop(teks, prob=0.1):
    """
    Hapus kata secara acak dengan probabilitas rendah.
    Mensimulasikan pesan yang typo atau tidak lengkap.
    Pastikan minimal 3 kata tersisa agar teks masih bermakna.
    """

    words = teks.split()
    if len(words) <= 3:
        return teks
    words_filtered = [w for w in words if random.random() > prob]
    if len(words_filtered) < 3:
        return teks
    return ' '.join(words_filtered)

def augmentasi_gabungan(teks):
    """
    Terapkan kombinasi teknik augmentasi secara acak.
    Setiap teknik diterapkan secara independen dengan probabilitas berbeda.
    """

    # Pilih kombinasi teknik secara acak
    teknik = random.choice([
        ['sinonim'],
        ['singkatan'],
        ['sinonim', 'singkatan'],
        ['sinonim', 'drop'],
        ['singkatan', 'drop'],
    ])

    hasil = teks
    if 'sinonim'   in teknik: hasil = augmentasi_sinonim(hasil)
    if 'singkatan' in teknik: hasil = augmentasi_singkatan(hasil)
    if 'drop'      in teknik: hasil = augmentasi_random_drop(hasil)
    return hasil

print('✅ Fungsi augmentasi siap')
print(f'\nContoh augmentasi:')
contoh = 'Selamat anda terpilih mendapat hadiah segera hubungi untuk klaim'
print(f'  Asli       : {contoh}')
print(f'  Sinonim    : {augmentasi_sinonim(contoh)}')
print(f'  Singkatan  : {augmentasi_singkatan(contoh)}')
print(f'  Gabungan 1 : {augmentasi_gabungan(contoh)}')
print(f'  Gabungan 2 : {augmentasi_gabungan(contoh)}')
print(f'  Gabungan 3 : {augmentasi_gabungan(contoh)}')

In [ ]:
# Terapkan augmentasi HANYA pada df_train
print('Data training sebelum augmentasi:')
print(df_train['label'].value_counts())
print(f'\nTotal: {len(df_train)} baris')

# Buat 1 salinan augmentasi dari seluruh data training
df_aug_1 = df_train.copy()
df_aug_1['teks']           = df_aug_1['teks'].apply(augmentasi_gabungan)
df_aug_1['teks_bersih']    = df_aug_1['teks'].apply(bersihkan_teks)
df_aug_1['teks_processed'] = df_aug_1['teks_bersih'].progress_apply(preprocess)
df_aug_1['sumber']         = df_aug_1['sumber'] + '_aug1'

# Buat salinan augmentasi kedua dengan variasi berbeda
df_aug_2 = df_train.copy()
df_aug_2['teks']           = df_aug_2['teks'].apply(augmentasi_gabungan)
df_aug_2['teks_bersih']    = df_aug_2['teks'].apply(bersihkan_teks)
df_aug_2['teks_processed'] = df_aug_2['teks_bersih'].progress_apply(preprocess)
df_aug_2['sumber']         = df_aug_2['sumber'] + '_aug2'

# Gabungkan: data asli + 2 augmentasi
df_train_aug = pd.concat(
    [df_train, df_aug_1, df_aug_2],
    ignore_index=True
)

# Hapus augmentasi yang persis sama dengan aslinya (jika ada)
sebelum_dedup = len(df_train_aug)
df_train_aug  = df_train_aug.drop_duplicates(subset='teks_processed').reset_index(drop=True)

print(f'\nData training SETELAH augmentasi:')
print(df_train_aug['label'].value_counts())
print(f'Total           : {len(df_train_aug)} baris')
print(f'\nDuplikat dibuang: {sebelum_dedup - len(df_train_aug)} baris')
print(f'\nPeningkatan data: {len(df_train)} → {len(df_train_aug)} ({len(df_train_aug)/len(df_train):.1f}x lipat)')
print(f'\nValidasi: df_val dan df_test TIDAK diaugmentasi')
print(f'\nValidasi/Test TIDAK diaugmentasi')

print('\nVALIDATION')
print(f'  df_val_sms   : {len(df_val_sms)} baris')
print(f'  df_val_email : {len(df_val_email)} baris')

print('\nTEST')
print(f'  df_test_sms   : {len(df_test_sms)} baris')
print(f'  df_test_email : {len(df_test_email)} baris')


In [ ]:
# Validasi hasil augmentasi — pastikan tidak ada leakage
print('='*55)
print('VALIDASI AUGMENTASI')
print('='*55)

# Cek distribusi label tetap seimbang
vc_aug = df_train_aug['label'].value_counts()
for lbl, cnt in vc_aug.items():
    nama = 'phishing' if lbl == 1 else 'normal'
    pct  = cnt / len(df_train_aug) * 100
    print(f'  Label {lbl} ({nama:<8}) : {cnt:>5} ({pct:.1f}%)')

rasio = vc_aug.get(0,0) / vc_aug.get(1,1)
print(f'  Rasio normal:phishing = {rasio:.2f}:1')

# Cek tidak ada overlap teks antara train_aug dan val/test

teks_val_sms = set(df_val_sms['teks_processed'].tolist())
teks_val_email = set(df_val_email['teks_processed'].tolist())
teks_test_sms = set(df_test_sms['teks_processed'].tolist())
teks_test_email = set(df_test_email['teks_processed'].tolist())
teks_train_aug = set(df_train_aug['teks_processed'].tolist())

overlap_val_sms = teks_train_aug & teks_val_sms
overlap_val_email = teks_train_aug & teks_val_email

overlap_test_sms = teks_train_aug & teks_test_sms
overlap_test_email = teks_train_aug & teks_test_email

print(f'\nCek data leakage:')

print('\nVALIDATION')
print(f'  train_aug ↔ val_sms    : {len(overlap_val_sms)} '
      f'{"✅ Aman" if len(overlap_val_sms) == 0 else "⚠️ Leakage!"}')

print(f'  train_aug ↔ val_email  : {len(overlap_val_email)} '
      f'{"✅ Aman" if len(overlap_val_email) == 0 else "⚠️ Leakage!"}')

print('\nTEST')
print(f'  train_aug ↔ test_sms   : {len(overlap_test_sms)} '
      f'{"✅ Aman" if len(overlap_test_sms) == 0 else "⚠️ Leakage!"}')

print(f'  train_aug ↔ test_email : {len(overlap_test_email)} '
      f'{"✅ Aman" if len(overlap_test_email) == 0 else "⚠️ Leakage!"}')


In [ ]:
# =========================
# MENANGANI DATA LEAKAGE
# Hapus overlap train_aug dengan val/test
# =========================

# Validation
teks_val_sms = set(df_val_sms['teks_processed'].tolist())
teks_val_email = set(df_val_email['teks_processed'].tolist())

# Test
teks_test_sms = set(df_test_sms['teks_processed'].tolist())
teks_test_email = set(df_test_email['teks_processed'].tolist())

# Gabungkan semua benchmark
teks_blocked = (
    teks_val_sms |
    teks_val_email |
    teks_test_sms |
    teks_test_email
)

sebelum_fix = len(df_train_aug)

# Hapus overlap dari train_aug
df_train_aug = df_train_aug[
    ~df_train_aug['teks_processed'].isin(teks_blocked)
].reset_index(drop=True)

print(f'Baris dihapus karena overlap: {sebelum_fix - len(df_train_aug)}')
print(f'df_train_aug setelah fix    : {len(df_train_aug)} baris')

In [ ]:
# =========================
# VERIFIKASI ULANG
# =========================

teks_train_aug = set(df_train_aug['teks_processed'].tolist())

overlap_val_sms = teks_train_aug & teks_val_sms
overlap_val_email = teks_train_aug & teks_val_email

overlap_test_sms = teks_train_aug & teks_test_sms
overlap_test_email = teks_train_aug & teks_test_email

print(f'\nVerifikasi setelah fix:')

print(f'  train_aug ↔ val_sms    : {len(overlap_val_sms)} '
      f'{"✅ Aman" if len(overlap_val_sms) == 0 else "⚠️ Leakage!"}')

print(f'  train_aug ↔ val_email  : {len(overlap_val_email)} '
      f'{"✅ Aman" if len(overlap_val_email) == 0 else "⚠️ Leakage!"}')

print(f'  train_aug ↔ test_sms   : {len(overlap_test_sms)} '
      f'{"✅ Aman" if len(overlap_test_sms) == 0 else "⚠️ Leakage!"}')

print(f'  train_aug ↔ test_email : {len(overlap_test_email)} '
      f'{"✅ Aman" if len(overlap_test_email) == 0 else "⚠️ Leakage!"}')

**Ringkasan Augmentasi:**

Proses augmentasi menghasilkan peningkatan data training yang signifikan tanpa menyentuh validation dan test set, sehingga evaluasi model tetap mencerminkan kondisi data nyata.

Teknik yang diterapkan:
- **Sinonim lokal** — variasi kata khas SMS Indonesia (40% probabilitas per kata)
- **Variasi singkatan** — konversi formal ↔ singkatan (35% probabilitas per kata)
- **Random word drop** — simulasi pesan tidak lengkap (10% probabilitas per kata)

Setiap data di-augmentasi 2x dengan kombinasi teknik yang berbeda secara acak, sehingga tiap salinan memiliki variasi unik. Duplikat dihapus setelah proses augmentasi selesai.


# **Export Dataset Final**

Dataset yang diekspor:
- `dataset_phishing_sms_indo.csv` — dataset lengkap (untuk referensi dan EDA)
- `dataset_train_aug.csv` — data training **setelah augmentasi** → dipakai AI untuk training model
- `dataset_val.csv` — data validasi (tidak diaugmentasi)
- `dataset_test.csv` — data test final (tidak diaugmentasi, tidak boleh dilihat selama training)

> **Catatan untuk AI Engineer:** Gunakan `dataset_train_aug.csv` untuk training, bukan `dataset_train.csv` yang belum diaugmentasi.


In [ ]:
# ✅ FIX 4: Val & Test — cap email agar proporsi seimbang dengan SMS
# Sebelumnya val=67% email, test=65% email → evaluasi bias ke domain email
# Fix: sama-kan rasio email:SMS di val/test seperti di train (max 50%)

def cap_email_split(df_sms, df_email, ratio=0.5, random_state=42):
    """
    Batasi email agar proporsinya = ratio dari total split.
    ratio=0.5 → email dan SMS jumlahnya sama.
    """
    n_sms   = len(df_sms)
    n_email = len(df_email)
    max_email = int((ratio / (1 - ratio)) * n_sms)
    max_email = min(max_email, n_email)

    if n_email > max_email:
        df_email = (
            df_email
            .groupby('label', group_keys=False)
            .apply(lambda x: x.sample(
                n=min(len(x), int(max_email * len(x) / n_email)),
                random_state=random_state
            ))
        )
    return pd.concat([df_sms, df_email]).sample(frac=1, random_state=random_state).reset_index(drop=True)

df_val  = cap_email_split(df_val_sms,  df_val_email)
df_test = cap_email_split(df_test_sms, df_test_email)

# Ringkasan
for df, name in [(df_val, 'val'), (df_test, 'test')]:
    n_em  = df['sumber'].str.contains('kaggle').sum()
    n_sms = len(df) - n_em
    vc    = df['label'].value_counts()
    print(f'[{name}] total={len(df)} | SMS={n_sms} ({n_sms/len(df)*100:.1f}%) | Email={n_em} ({n_em/len(df)*100:.1f}%)')
    print(f'  label 0={vc.get(0,0)} label 1={vc.get(1,0)}')


In [ ]:
# =========================
# EXPORT DATASET
# =========================

# Dataset lengkap
df_final.to_csv('dataset_phishing_sms_indo.csv', index=False, encoding='utf-8-sig')

# Train augmented
df_train_aug.to_csv('dataset_train_aug.csv', index=False, encoding='utf-8-sig')

# Train original
df_train.to_csv('dataset_train_original.csv', index=False, encoding='utf-8-sig')

# Validation per domain
df_val_sms.to_csv('dataset_val_sms.csv',     index=False, encoding='utf-8-sig')
df_val_email.to_csv('dataset_val_email.csv', index=False, encoding='utf-8-sig')

# Test per domain
df_test_sms.to_csv('dataset_test_sms.csv',     index=False, encoding='utf-8-sig')
df_test_email.to_csv('dataset_test_email.csv', index=False, encoding='utf-8-sig')

# Val & test gabungan (sudah di-cap email-nya)
df_val.to_csv('dataset_val.csv',   index=False, encoding='utf-8-sig')
df_test.to_csv('dataset_test.csv', index=False, encoding='utf-8-sig')

print('✅ File berhasil disimpan')
print(f'\n{"="*55}')
print('RINGKASAN FINAL')
print(f'{"="*55}')

def ringkasan_split(df, name):
    n_em  = df['sumber'].str.contains('kaggle', na=False).sum()
    n_sms = len(df) - n_em
    vc    = df['label'].value_counts()
    print(f'\n[{name}] total={len(df)}')
    print(f'  SMS asli : {n_sms} ({n_sms/len(df)*100:.1f}%)')
    print(f'  Email    : {n_em}  ({n_em/len(df)*100:.1f}%)')
    print(f'  label 0  : {vc.get(0,0)} | label 1 : {vc.get(1,0)}')

ringkasan_split(df_train_aug, 'train_aug')
ringkasan_split(df_val,       'val')
ringkasan_split(df_test,      'test')

print(f'\n✅ Proporsi email di semua split sudah seimbang (<= 50%)')
print(f'✅ Siap untuk tahap training dan evaluasi')


In [ ]:
train = pd.read_csv('dataset_train_aug.csv')

print("Proporsi data ASLI (non-augmented, non-email):")
sms_asli = train[train['sumber'].isin(['yudiwbs_gist', 'bopbi_loan', 'bopbi_prize',
                                        'bopbi_premium-sms', 'bopbi_evil-service',
                                        'bopbi_non-provider-promo'])]
print(f"SMS Indonesia asli : {len(sms_asli)} baris ({len(sms_asli)/len(train)*100:.1f}%)")
print(f"Email (semua)      : {len(train[train['sumber'].str.contains('kaggle')])} baris")